In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [4]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

model_with_tools = model.bind_tools([multiply])

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[multiply],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "chat-1"}}

agent.invoke({"messages": [{"role": "user", "content": "My name is Ashik."}]}, config)
out = agent.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(out["messages"][-1].content)

Your name is Ashik.


In [6]:
# See each step as it completes
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is 47 times 89?"}]},
    config, stream_mode="updates",
):
    print(chunk)

# Token-by-token text
for token, meta in agent.stream(
    {"messages": [{"role": "user", "content": "Explain RAG briefly."}]},
    config, stream_mode="messages",
):
    print(token.content, end="")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': "We need to compute 47 * 89. Let's calculate: 47*89 = 47*(90-1)=47*90 - 47 = 4230 - 47 = 4183. Could also use multiply function. Use tool.", 'tool_calls': [{'id': 'fc_9e369202-dbc9-4bfd-ac49-0c1e5ea401dc', 'function': {'arguments': '{"a":47,"b":89}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 180, 'total_tokens': 266, 'completion_time': 0.183499327, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.007952431, 'prompt_tokens_details': None, 'queue_time': 0.318915454, 'total_time': 0.191451758}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0c366-5204-7703-8d73-fdbb490d097d-0', tool_calls=[{'name': 'multiply', 'args': {'a': 47, 'b': 89}, 'id': 'fc_9e369202-d